In [1]:
import pandas as pd
import re
import requests
import time
import json

In [2]:
# Read in the main resale dataset
df = pd.read_csv("Resale flat prices based on registration date from Jan-2017 onwards.csv")

In [3]:
# Keep only block and street name columns
df = df[["block", "street_name"]]

# Combine block and street name into a single full address column
df["Location"] = df["block"] + " " + df["street_name"]

# Save the result to a CSV file
df.to_csv("location_combined.csv", index=False)

## Extract Coordinates Using OneMap API 

In [4]:
# === CONFIGURATION ===
INPUT_FILE = 'location_combined.csv'
OUTPUT_FILE = 'location_with_latlong.csv'

In [5]:
# === ONEMAP QUERY FUNCTION ===

# Query latitude and longitude from OneMap API based on an address
def get_coords_onemap(address):
    
    # Try up to 3 times in case of temporary request issues
    for _ in range(3):
        try:
            
            # Short pause to avoid hitting the API too quickly
            time.sleep(0.3)   
            url = "https://www.onemap.gov.sg/api/common/elastic/search"
            params = {
                "searchVal": address,
                "returnGeom": "Y",
                "getAddrDetails": "Y",
                "pageNum": 1
            }
            
            # Send GET request to OneMap API
            response = requests.get(url, params=params, timeout=60)
    
            # Return the first matched result if found
            data = response.json()
            if data["found"] > 0:
                result = data["results"][0]
                return f"{result['LATITUDE']}, {result['LONGITUDE']}"
            
        # Print error message if request fails
        except Exception as e:
            print("Error:", address, e)
            
        # Return NIL if no result is found after all attempts
        return "NIL"

# Read input file
df = pd.read_csv(INPUT_FILE)

# Apply OneMap geocoding function to each location
df['Lat Long'] = df['Location'].apply(get_coords_onemap)

# Save output file with coordinates
df.to_csv(OUTPUT_FILE, index=False)
print("Done!")

Done!
